In [2]:
def modInverse(a, m):
    # Standard modular inverse using Fermat's Little Theorem (works for prime m)
    return pow(a, m - 2, m)

def ntt(a, q, omega):
    """
    Recursive Number-Theoretic Transform.
    a: input list (length must be a power of 2)
    q: prime modulus
    omega: n-th primitive root of unity modulo q
    """
    n = len(a)
    if n == 1:
        return a

    # Split into even and odd indices (standard Cooley-Tukey)
    a_even = ntt(a[0::2], q, pow(omega, 2, q))
    a_odd = ntt(a[1::2], q, pow(omega, 2, q))

    # Combine results
    combined = [0] * n
    twiddle = 1
    for k in range(n // 2):
        # The core "butterfly" operation
        t = (twiddle * a_odd[k]) % q
        combined[k] = (a_even[k] + t) % q
        combined[k + n // 2] = (a_even[k] - t) % q
        twiddle = (twiddle * omega) % q
        
    return combined

def intt(a_ntt, q, omega):
    """
    Inverse Number-Theoretic Transform.
    a_ntt: list of NTT coefficients
    q: prime modulus
    omega: n-th primitive root of unity used in the forward NTT
    """
    n = len(a_ntt)
    
    # 1. Find the inverse root: omega_inv
    omega_inv = modInverse(omega, q)
    
    # 2. Perform a standard NTT using the inverse root
    # Note: We reuse the same logic as the forward NTT
    res = ntt(a_ntt, q, omega_inv)
    
    # 3. Normalize by multiplying by n_inv
    n_inv = modInverse(n, q)
    original_poly = [(val * n_inv) % q for val in res]
    
    return original_poly

# --- Example Continued ---
# Let's use the NTT result from the previous demo: [3, 10, 16, 9]
q = 17
omega = 13
ntt_coefficients = [3, 10, 16, 9]

# Perform the Inverse Transform
recovered_poly = intt(ntt_coefficients, q, omega)

print(f"NTT Coefficients: {ntt_coefficients}")
print(f"Recovered Original Coefficients: {recovered_poly}")

NTT Coefficients: [3, 10, 16, 9]
Recovered Original Coefficients: [1, 2, 0, 0]
